# 🚗 BẢN 1: HUẤN LUYỆN & TÁI HIỆN BASELINE (FOCALCOMM WACV 2026)
### 📘 Mô Hình Cơ Sở Đối Chứng: Hard Instance-Aware Multi-Agent Perception (HIM + QAFF)

Notebook này dùng để chạy **Bản Baseline nguyên bản** của bài báo FocalComm (WACV 2026):
- **Kiến trúc:** HIM (Hard Instance Mining) + QAFF (Query-guided Adaptive Feature Fusion).
- **Mốc đối chứng (Benchmark):** Tái hiện kết quả **56.1% mAP** trên bộ dữ liệu V2X-Real.
- **Mã nguồn:** Clone trực tiếp từ GitHub repo: [tranvuquanghuy12/FocalComm](https://github.com/tranvuquanghuy12/FocalComm.git).

---
### ⚙️ Hướng Dẫn Thực Hiện:
1. **Chế độ 1 (Nghiệm thu nhanh bằng Checkpoint chính thức):** Tải weights của tác giả từ Hugging Face và đánh giá để ra ngay bảng số liệu đối chứng 56.1% mAP.
2. **Chế độ 2 (Huấn luyện lại Baseline từ đầu):** Chạy `train_focalcomm.py` với cấu hình `focalcommv3_kaggle.yaml`.

## 1. Kiểm tra môi trường GPU trên Kaggle
Yêu cầu bật Accelerator: GPU (T4 hoặc P100) và Internet: ON.

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Hãy bật Accelerator = GPU ở menu bên phải Kaggle!")

!nvidia-smi


## 2. Clone Repository FocalComm từ GitHub
Lấy mã nguồn Baseline mới nhất.

In [ ]:
# Xóa thư mục cũ nếu có để đảm bảo lấy code mới nhất
!rm -rf /kaggle/working/FocalComm
!git clone https://github.com/tranvuquanghuy12/FocalComm.git /kaggle/working/FocalComm

%cd /kaggle/working/FocalComm
!git log -n 1 --oneline


## 3. Cài đặt các thư viện cần thiết & Biên dịch CUDA Extensions

In [ ]:
# 1. Thư viện cơ bản
!pip install -q easydict numba open3d cython tensorboardX shapely==2.0.0
!pip install -q pyquaternion einops timm icecream python-lzf termcolor efficientnet_pytorch==0.7.0 huggingface_hub
!pip install -q git+https://github.com/klintan/pypcd.git

# 2. Cài spconv cho CUDA
!pip install -q spconv-cu118 || pip install -q spconv-cu120 || pip install -q spconv-cu117

# 3. Biên dịch CUDA & Cython Extensions
%cd /kaggle/working/FocalComm/focalcomm/utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm/focalcomm/pcdet_utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm
print("✅ Môi trường và CUDA Ops của Baseline đã sẵn sàng!")


## 4. Thiết lập Bộ Dữ Liệu V2X-Real

In [ ]:
import os
DATASET_ROOT = "/kaggle/working/dataset/V2XReal"
os.makedirs(DATASET_ROOT, exist_ok=True)

# Tự động phát hiện dữ liệu nếu bạn đã add dataset trên Kaggle
kaggle_input_dirs = [d for d in os.listdir("/kaggle/input") if "v2x" in d.lower() or "focalcomm" in d.lower()] if os.path.exists("/kaggle/input") else []
if kaggle_input_dirs:
    src = os.path.join("/kaggle/input", kaggle_input_dirs[0])
    print(f"Phát hiện dataset trong Kaggle Input: {src}")
    if not os.path.exists(os.path.join(DATASET_ROOT, "train")) and os.path.exists(os.path.join(src, "train")):
        os.symlink(os.path.join(src, "train"), os.path.join(DATASET_ROOT, "train"))
    if not os.path.exists(os.path.join(DATASET_ROOT, "val")) and os.path.exists(os.path.join(src, "val")):
        os.symlink(os.path.join(src, "val"), os.path.join(DATASET_ROOT, "val"))
    print(f"Liên kết dữ liệu thành công vào: {DATASET_ROOT}")
else:
    print(f"Dataset sẽ được đọc từ: {DATASET_ROOT}")


## 5. LỰA CHỌN A: Đánh Giá Nghiệm Thu Baseline Ngay Lập Tức (56.1% mAP)
Tải checkpoint chính thức của tác giả FocalComm và đánh giá đối chứng trên tập V2X-Real.

In [ ]:
from huggingface_hub import snapshot_download
import os

ckpt_dir = "checkpoints/focalcomm_v2xreal"
os.makedirs(ckpt_dir, exist_ok=True)

print("Đang tải checkpoint chính thức của Baseline từ Hugging Face (scdrand23/FocalComm)...")
snapshot_download(
    repo_id="scdrand23/FocalComm",
    local_dir="checkpoints",
    allow_patterns=["*focalcomm_v2xreal*", "*net_epoch_best.pth*"]
)
print("✅ Đã tải checkpoint Baseline thành công!")

# Chạy đánh giá trên tập kiểm thử
!python focalcomm/tools/inference.py \
    --model_dir checkpoints/focalcomm_v2xreal \
    --fusion_method intermediate


## 6. LỰA CHỌN B: Huấn Luyện Lại Baseline từ đầu (30 Epochs)
Sử dụng cấu hình tối ưu VRAM cho Kaggle GPU (`focalcommv3_kaggle.yaml`).

In [ ]:
# Huấn luyện Baseline FocalComm gốc với cơ chế cộng tác HIM + QAFF
!python focalcomm/tools/train_focalcomm.py \
    --hypes_yaml focalcomm/hypes_yaml/v2xreal/focalcommv3_kaggle.yaml


## 7. Đóng Gói Checkpoints Baseline để tải về

In [ ]:
output_tar = "/kaggle/working/baseline_focalcomm_results.tar.gz"
!tar -czvf {output_tar} checkpoints/
print(f"✅ Đã tạo file nén Baseline tại: {output_tar}")
